#### 철강 공장에서 만든 강판(철판) 표면에는 여러 종류의 흠(결함)이 생긴다. 카메라로 결함을 찍어서 **위치, 크기, 둘레, 밝기 같은 숫자 27개**를 측정해 두었는데, 이 숫자만 보고 그 결함이 **7가지 유형 중 무엇인지** 자동으로 맞히는 프로그램을 만드는 것이 목표다.

| 결함 유형 | 개수(전체 1,941건 중) | 비고 |
|---|---|---|
| Other_Faults (기타) | 673 | 가장 많음 |
| Bumps (돌기) | 402 | |
| K_Scatch (긁힘 K형) | 391 | |
| Z_Scratch (긁힘 Z형) | 190 | |
| Pastry (얼룩 반죽형) | 158 | |
| Stains (얼룩) | 72 | |
| Dirtiness (오염) | 55 | 가장 적음 |

이렇게 **"정답이 붙어 있는 과거 데이터로 규칙을 배운 뒤, 새 데이터의 정답을 맞히는 것"**을 **지도학습**, 그중 정답이 '종류'인 경우를 **분류(classification)**라고 한다. 위 표처럼 유형별 개수 차이가 큰 것을 **클래스 불균형**이라고 하며, 이 코드의 여러 설계가 이 불균형 때문에 나왔다.

## 먼저 알아둘 용어

| 용어 | 뜻 | 비유 |
|---|---|---|
| 행(샘플) | 결함 1건 | 문제 1개 |
| 변수(특징, feature) | 측정값 한 종류 (표의 열) | 문제에 주어진 단서 |
| 타깃(정답, 레이블) | 결함 유형 | 문제의 답 |
| 모델 | 단서 → 답의 규칙을 데이터에서 스스로 찾는 프로그램 | 학생 |
| 학습(fit) | 모델이 규칙을 찾는 과정 | 문제집 풀며 공부 |
| 예측(predict) | 학습한 규칙으로 새 데이터의 답을 내는 것 | 시험 보기 |
| 과적합(overfitting) | 학습 데이터의 우연한 특징까지 외워서 새 데이터에서는 못 맞히는 상태 | 문제집 답만 외운 학생 |
| 하이퍼파라미터 | 학습 **전에** 사람이 정하는 모델 설정값 | 공부 방법(하루 몇 시간, 몇 회독) |
| 테스트셋 | 마지막에 딱 한 번 성능을 재는, 학습에 절대 쓰지 않는 데이터 | 수능 |
| 교차검증 | 학습 데이터를 여러 조각으로 나눠 돌아가며 연습 시험을 보는 방법 | 모의고사 여러 번 |

## 전체 흐름

1. 데이터 불러오기 및 형태 정리 → 2. 강종과 결함의 관계 확인 → 3. 입력 변수 만들기 → 4. 개발용/테스트용 나누기 → 5. 추가 변수를 쓸지 결정 → 6. 사용할 모델과 설정 후보 정하기 → 7. 모델별 최적 설정 찾기 → 8. 앙상블·확률 보정 후보 비교 → 9. 최종 모델 결정 → 10. 테스트 평가 → 11. 틀린 사례 분석 → 12. 어떤 변수가 중요했는지 → 13. 전체 데이터로 다시 학습·저장 → 14. 새 데이터 예측 함수

> **이 코드의 출신**: 같은 과제를 GPT와 Claude가 각각 작성한 두 코드에서 장점을 합친 버전이다.
> GPT 쪽에서는 *교차검증 기반 튜닝, 클래스 가중치 튜닝, 여러 모델 비교*를, Claude 쪽에서는 *데이터 구조 검증, 통계 검정, 변수 중요도, 오분류 분석, 배포용 예측 함수*를 가져왔고, *파생변수 채택 검증, 앙상블, 클래스별 확률 보정, 신뢰구간*을 새로 넣었다.

> **가장 중요한 원칙 하나**: 테스트 데이터는 마지막에 **딱 한 번**만 본다. 테스트 점수를 보고 모델을 고치면, 수능 문제를 미리 보고 공부한 것과 같아서 점수가 실제 실력보다 부풀려진다. 그래서 4단계에서 테스트를 떼어 놓은 뒤 9단계까지의 모든 결정은 개발 데이터만으로 내린다.

---
## 0단계. 라이브러리와 기본 설정

### 사용하는 도구(라이브러리)
| 라이브러리 | 하는 일 |
|---|---|
| `pandas` | 엑셀 같은 표(DataFrame)를 다루는 도구 |
| `numpy` | 숫자 배열 계산 |
| `matplotlib` | 그래프 그리기 |
| `scipy` | 통계 검정(카이제곱 검정) |
| `scikit-learn` (`sklearn`) | 머신러닝 도구 상자: 데이터 나누기, 교차검증, 모델, 평가지표 |
| `lightgbm`, `catboost` | 성능이 좋은 '부스팅' 모델 (6단계에서 설명) |
| `joblib` | 학습한 모델을 파일로 저장/불러오기 |

### 성능을 어떻게 잴 것인가 — 평가지표
모델이 얼마나 잘 맞히는지 숫자로 재는 기준이다. 결함 하나(예: Dirtiness)를 기준으로 보면:

- **정밀도(Precision)**: 모델이 "Dirtiness다"라고 한 것 중 진짜 Dirtiness의 비율. → *헛경보가 적은가?*
- **재현율(Recall)**: 진짜 Dirtiness 중 모델이 찾아낸 비율. → *놓치는 게 적은가?*
- **F1 점수**: 정밀도와 재현율의 조화평균. 둘 중 하나만 높으면 낮게 나오므로, 둘 다 좋아야 높다.
- **정확도(Accuracy)**: 전체 중 맞힌 비율.

**왜 정확도가 아니라 Macro F1을 주 지표로 쓰나?**
정확도는 개수가 많은 유형에 끌려간다. Other_Faults(673건)를 잘 맞히면 Dirtiness(55건)를 전부 틀려도 정확도는 크게 안 떨어진다. 하지만 공장 입장에서는 드문 결함도 중요하다.
**Macro F1**은 7개 유형의 F1을 각각 구한 뒤 **단순 평균**을 낸다. 개수와 상관없이 모든 유형을 똑같이 1/7씩 반영하므로, 드문 유형을 무시하는 모델은 낮은 점수를 받는다.
**Balanced Accuracy**는 유형별 재현율의 평균으로, 같은 취지의 보조 지표다.

In [ ]:
# ============================================================
# 0. 라이브러리 및 설정
# ============================================================

import time          # 튜닝에 걸린 시간 측정
import warnings      # 불필요한 경고 메시지 숨기기

import joblib        # 모델 저장/불러오기
import numpy as np   # 숫자 계산
import pandas as pd  # 표 데이터 다루기
import matplotlib.pyplot as plt  # 그래프

# 주피터 노트북에서는 표를 예쁘게 보여주는 display를, 아니면 print를 사용
try:
    from IPython.display import display
except ImportError:
    display = print

from scipy.stats import chi2_contingency              # 카이제곱 검정 (2단계)
from sklearn.base import BaseEstimator, ClassifierMixin, clone  # 나만의 모델 클래스 만들기, 모델 복제
from sklearn.model_selection import (
    train_test_split,         # 데이터를 개발용/테스트용으로 나누기
    StratifiedKFold,          # 층화 K-겹 교차검증
    RepeatedStratifiedKFold,  # 위 교차검증을 여러 번 반복
    RandomizedSearchCV,       # 하이퍼파라미터 무작위 탐색
    cross_val_score,          # 교차검증 점수 계산
    cross_val_predict,        # 교차검증 방식의 예측값(Out-of-fold) 계산
)
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.inspection import permutation_importance  # 변수 중요도 (12단계)
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
)
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)   # 표를 출력할 때 열을 생략하지 않음
pd.set_option("display.width", 200)

plt.rcParams["font.family"] = "Malgun Gothic"   # 그래프 한글 깨짐 방지 (macOS: "AppleGothic")
plt.rcParams["axes.unicode_minus"] = False      # 마이너스 기호 깨짐 방지

# 무작위 요소(데이터 섞기, 모델 내부의 무작위 선택 등)의 '씨앗' 값.
# 이 숫자를 고정하면 코드를 몇 번 다시 돌려도 결과가 똑같이 나온다(재현성).
# 42라는 값 자체에는 의미가 없다.
RANDOM_STATE = 42
DATA_PATH = "steel_plates_faults.csv"

# 7단계에서 모델마다 시험해 볼 설정 조합의 개수.
# 많을수록 좋은 설정을 찾을 확률이 높지만 시간이 비례해서 늘어난다.
N_ITER = 20            # 모델별 무작위 탐색 조합 수 (시간이 부족하면 줄이기)

# 원본 파일에서 강종(철 종류)과 결함 유형은 '원-핫' 형태(여러 열에 0/1)로 저장되어 있다.
STEEL_COLS = ["TypeOfSteel_A300", "TypeOfSteel_A400"]
FAULT_COLS = ["Pastry", "Z_Scratch", "K_Scatch", "Stains",
              "Dirtiness", "Bumps", "Other_Faults"]
TARGET = "Fault_Type"   # 정답(맞혀야 할 값) 열 이름

# 모델은 글자가 아닌 숫자를 다루므로 결함 이름 ↔ 번호(0~6) 변환표를 만든다.
LABELS = list(range(len(FAULT_COLS)))
class_to_int = {name: i for i, name in enumerate(FAULT_COLS)}
int_to_class = {i: name for name, i in class_to_int.items()}
STEEL_CODE = {"A300": 0, "A400": 1}


def macro_f1(y_true, y_pred):
    """7개 유형의 F1을 각각 구해 단순 평균 (드문 유형도 똑같이 반영).
    zero_division=0: 어떤 유형을 한 번도 예측하지 않아 정밀도를 계산할 수 없으면 0점 처리."""
    return f1_score(y_true, y_pred, labels=LABELS, average="macro", zero_division=0)


def score_row(y_true, y_pred):
    """여러 평가지표를 한 번에 계산해 딕셔너리로 반환"""
    return {
        "Accuracy": accuracy_score(y_true, y_pred),               # 전체 정확도
        "Balanced Acc": balanced_accuracy_score(y_true, y_pred),  # 유형별 재현율 평균
        "Macro Precision": precision_score(y_true, y_pred, labels=LABELS,
                                           average="macro", zero_division=0),
        "Macro Recall": recall_score(y_true, y_pred, labels=LABELS,
                                     average="macro", zero_division=0),
        "Macro F1": macro_f1(y_true, y_pred),                     # 주 지표
    }

---
## 1단계. 데이터 불러오기와 열 병합

### 원-핫(one-hot) 형식이란?
원본 파일은 "이 강판은 어떤 강종인가"를 한 열에 쓰지 않고, 아래처럼 **종류마다 열을 하나씩 두고 해당하는 곳에만 1**을 적는다.

| TypeOfSteel_A300 | TypeOfSteel_A400 | → | Steel_Type |
|---|---|---|---|
| 1 | 0 | → | A300 |
| 0 | 1 | → | A400 |

결함 유형도 7개 열이 같은 방식이다. 이 코드는 이를 **한 열로 합친다**.
- 강종 2열 → `Steel_Type` 1열
- 결함 7열 → `Fault_Type` 1열 (정답 열)

**왜 합치나?** 정답이 7개 열로 흩어져 있으면 모델에 "정답은 이것"이라고 하나로 건네기 어렵고, 사람이 표를 읽기도 불편하다.

### 왜 먼저 '구조 검증'을 하나?
합치기 전에 "각 행에서 정확히 한 열만 1인가"를 확인한다. 만약 결함이 두 개 동시에 1이거나 하나도 없는 행이 있다면, 합칠 때 조용히 엉뚱한 값이 들어간다. 이런 문제는 나중에 발견하기 매우 어려우므로, **처음에 확인하고 문제가 있으면 바로 멈추게(에러)** 한다.

In [ ]:
# ============================================================
# 1. 데이터 불러오기 및 열 병합 (원-핫 구조 검증 포함)
# ============================================================

raw_df = pd.read_csv(DATA_PATH)
print("원본 데이터 크기:", raw_df.shape)   # (행 수, 열 수)
# 결측치(빈칸)와 완전히 똑같은 행이 있는지 확인 → 이 데이터는 둘 다 0개여야 정상
print("결측치:", raw_df.isna().sum().sum(), "/ 중복 행:", raw_df.duplicated().sum())

# 강종 열과 결함 열 각각에 대해 세 가지를 검사한다.
for cols, label in [(STEEL_COLS, "강종"), (FAULT_COLS, "결함 유형")]:
    # ① 필요한 열이 파일에 다 있는가
    missing = [c for c in cols if c not in raw_df.columns]
    if missing:
        raise ValueError(f"{label} 열이 없습니다: {missing}")
    # ② 값이 0 또는 1뿐인가
    if not raw_df[cols].isin([0, 1]).all().all():
        raise ValueError(f"{label} 열은 0 또는 1이어야 합니다.")
    # ③ 각 행에서 1이 정확히 하나인가 (가로 합이 1)
    if not (raw_df[cols].sum(axis=1) == 1).all():
        raise ValueError(f"{label}이(가) 정확히 하나로 정해지지 않는 행이 있습니다.")


def merge_columns(data: pd.DataFrame) -> pd.DataFrame:
    """강종 2열 → Steel_Type, 결함 7열 → Fault_Type (결함 열이 없으면 강종만)
    14단계의 새 데이터 예측에서도 재사용하므로, 결함 열이 없는 데이터도 처리할 수 있게 만들었다."""
    out = data.copy()   # 원본을 건드리지 않도록 복사본에서 작업
    if set(STEEL_COLS).issubset(out.columns):
        pos = out.columns.get_loc("TypeOfSteel_A300")            # 원래 열 위치 기억
        steel = np.where(out["TypeOfSteel_A300"] == 1, "A300", "A400")
        out = out.drop(columns=STEEL_COLS)
        out.insert(pos, "Steel_Type", steel)                     # 같은 자리에 새 열 삽입
    if set(FAULT_COLS).issubset(out.columns):
        # idxmax: 각 행에서 가장 큰 값(=1)이 있는 열의 '이름'을 돌려준다 → 그 이름이 결함 유형
        out[TARGET] = out[FAULT_COLS].idxmax(axis=1)
        out = out.drop(columns=FAULT_COLS)
    return out


df = merge_columns(raw_df)
print("병합 후 데이터 크기:", df.shape)

# 결함 유형별 개수 → 클래스 불균형 확인
print("\n[결함 유형 분포]")
display(df[TARGET].value_counts().reindex(FAULT_COLS).to_frame("count"))

---
## 2단계. 강종과 결함 유형의 관계 — 카이제곱 검정, Cramér's V

### 무엇을 알고 싶은가
"A300 강판과 A400 강판에서 생기는 결함 유형이 다른가?" 만약 다르다면, 강종 정보가 결함을 맞히는 데 도움이 되는 단서라는 뜻이다.

### 사용한 기법
1. **교차표(crosstab)**: 결함 유형(행) × 강종(열)로 개수를 센 표.
2. **카이제곱 독립성 검정**: "강종과 결함 유형이 서로 아무 관련이 없다면 교차표가 이런 모양이어야 한다"는 기대값과 실제 표를 비교해, 차이가 **우연으로 생길 만한 수준인지** 판단한다.
   - **p값**: "실제로는 관련이 없는데 우연히 이 정도 차이가 날 확률". 보통 0.05보다 작으면 '우연이라고 보기 어렵다 = 관련이 있다'고 해석한다.
3. **Cramér's V**: p값은 "관련이 있느냐 없느냐"만 말해 주고 **얼마나 강한지**는 말해 주지 않는다(데이터가 많으면 아주 약한 관련도 p값이 작아진다). Cramér's V는 관련의 강도를 0(무관)~1(완전히 결정됨)로 나타낸다. 대략 0.1 약함, 0.3 중간, 0.5 이상 강함으로 읽는다.

### 왜 이 기법인가
두 변수가 모두 **범주형**(종류로 나뉘는 값)일 때 관련성을 보는 가장 표준적인 방법이 카이제곱 검정이고, 그 강도를 보완하는 짝이 Cramér's V다.

> 참고: 이 단계는 데이터를 '구경'하는 설명용 분석이라 전체 데이터를 쓴다. 이 결과로 모델을 바꾸지는 않으므로(강종은 어차피 입력에 포함) 테스트 데이터 원칙을 어기지 않는다.

In [ ]:
# ============================================================
# 2. 강종 - 결함 유형 관계 (카이제곱 검정, Cramér's V)
# ============================================================

ct = pd.crosstab(df[TARGET], df["Steel_Type"]).reindex(FAULT_COLS)   # 교차표
chi2, p_value, dof, _ = chi2_contingency(ct)   # 카이제곱 통계량, p값, 자유도
# Cramér's V = sqrt( 카이제곱 / (전체 개수 × (행·열 중 작은 쪽 개수 - 1)) )
cramers_v = np.sqrt(chi2 / (len(df) * (min(ct.shape) - 1)))

print("\n[결함 유형 × 강종]")
display(ct)
print(f"Chi2 = {chi2:.1f}, df = {dof}, p = "
      f"{'<0.001' if p_value < 0.001 else round(p_value, 4)}, Cramér's V = {cramers_v:.3f}")

---
## 3단계. 입력 변수 만들기 — 파생변수(Feature Engineering)

### 파생변수란?
원래 있는 측정값을 계산해서 새로 만든 변수다. 예를 들어 '가로 길이'와 '세로 길이'가 있으면 '가로÷세로(길쭉한 정도)'를 만들 수 있다.

### 왜 일부 파생변수는 뺐나 — 트리 모델의 특성
이 코드에서 쓰는 모델은 모두 **의사결정나무(트리)** 계열이다(6단계). 트리는 "면적이 300보다 큰가?" 같은 **크기 비교 질문**으로 데이터를 나눈다.
그래서 어떤 변수에 로그를 씌우거나 일정한 배수를 곱하는 등 **순서를 바꾸지 않는 변환(단조 변환)**은 트리에게 새 정보가 없다. "면적 > 300"이나 "log(면적) > 5.7"이나 똑같이 나누기 때문이다.
- `X_Range`는 이미 있는 `Log_X_Index`와, `Y_Range`는 `Log_Y_Index`와, `Mean_Luminosity`는 `Luminosity_Index`와 이런 관계라서 **제외**했다.

### 후보로 남긴 3개 (새 정보가 생기는 것)
트리는 한 번에 **한 변수만** 보고 질문하기 때문에, **두 변수를 빼거나 나눈 값**은 스스로 만들어 내기 어렵다. 그래서 이런 조합은 직접 만들어 주면 도움이 될 수 있다.

| 파생변수 | 계산 | 의미 | 기대 효과 |
|---|---|---|---|
| `Luminosity_Range` | 최대 밝기 − 최소 밝기 | 결함 안의 밝기 대비 | 얼룩·오염처럼 색이 고르지 않은 결함 구분 |
| `Aspect_Ratio` | 가로 길이 ÷ 세로 길이 | 길쭉한 정도 | 긁힘(Scratch)은 한 방향으로 길쭉함 |
| `Perimeter_Area_Ratio` | (가로 둘레 + 세로 둘레) ÷ 면적 | 모양이 얼마나 울퉁불퉁/가는지 | 가늘고 복잡한 결함일수록 큼 |

> 길이에 `+1`을 하는 이유: 픽셀 좌표 5부터 5까지면 실제로는 픽셀 1칸이므로 `최대-최소+1`이 정확한 길이다. 덕분에 길이가 0이 되어 0으로 나누는 오류도 생기지 않는다.

이 3개를 **실제로 쓸지 말지는 사람이 감으로 정하지 않고** 5단계에서 데이터로 검증해서 정한다.

### 강종을 0/1로 바꾸는 이유
모델은 'A300' 같은 글자를 계산할 수 없으므로 A300=0, A400=1로 바꾼다. 값이 두 종류뿐이라 이렇게 바꿔도 의미가 왜곡되지 않는다.

In [ ]:
# ============================================================
# 3. 입력 변수 생성
# ============================================================
# Claude 코드의 파생변수 중 원본과 단조 관계라 트리 모델에 정보가 없는
# X_Range(≈Log_X_Index), Y_Range(≈Log_Y_Index), Mean_Luminosity(≈Luminosity_Index)는 제외.
# 정보가 새로 생기는 3개만 후보로 두고, 쓸지 말지는 5단계 교차검증으로 결정한다.
# ============================================================

EXTRA_FEATURES = ["Luminosity_Range", "Aspect_Ratio", "Perimeter_Area_Ratio"]


def build_features(data: pd.DataFrame, use_extra: bool) -> pd.DataFrame:
    """모델에 넣을 입력 표(X)를 만든다. use_extra=True면 파생변수 3개를 추가."""
    # 정답 열은 입력에 들어가면 안 된다(답을 보고 시험 보는 것과 같음) → 제거
    out = data.drop(columns=[TARGET], errors="ignore").copy()
    out["Steel_Type"] = out["Steel_Type"].map(STEEL_CODE)   # A300→0, A400→1
    if out["Steel_Type"].isna().any():   # 예상하지 못한 강종 값이 들어오면 멈춤
        raise ValueError("Steel_Type에 A300 / A400 이외의 값이 있습니다.")

    if use_extra:
        x_len = out["X_Maximum"] - out["X_Minimum"] + 1   # 결함의 가로 길이(픽셀)
        y_len = out["Y_Maximum"] - out["Y_Minimum"] + 1   # 결함의 세로 길이(픽셀)
        out["Luminosity_Range"] = out["Maximum_of_Luminosity"] - out["Minimum_of_Luminosity"]
        out["Aspect_Ratio"] = x_len / y_len
        out["Perimeter_Area_Ratio"] = (
            (out["X_Perimeter"] + out["Y_Perimeter"]) / out["Pixels_Areas"]
        )
    return out


# 정답을 결함 이름 → 번호(0~6)로 바꾼 배열
y_all = df[TARGET].map(class_to_int).to_numpy()

---
## 4단계. 개발용 / 테스트용 데이터 나누기

### 왜 나누나?
모델이 공부한 문제로 시험을 보면 당연히 점수가 높게 나온다. **처음 보는 데이터**에서 얼마나 맞히는지를 알아야 실제 공장에 썼을 때의 성능을 추정할 수 있다. 그래서 20%를 **테스트셋**으로 떼어 금고에 넣어 두고, 10단계에서 한 번만 꺼낸다.

### 왜 80 / 20인가?
흔히 쓰는 비율이다. 테스트를 너무 작게 하면 점수가 운에 많이 흔들리고, 너무 크게 하면 학습할 데이터가 줄어든다. 데이터가 1,941건이라 20%면 테스트 약 389건이다.

### 층화(stratify)란?
무작위로 나누면 운 나쁘게 Dirtiness(55건)가 테스트에 거의 안 들어갈 수도 있다. `stratify`를 쓰면 **개발·테스트 양쪽의 결함 유형 비율이 원본과 같게** 나눈다. 드문 유형이 있는 데이터에서는 거의 필수다.

### 교차검증(Cross Validation)이란?
개발 데이터 안에서도 "모델 A와 B 중 뭐가 나은가"를 판단하려면 연습 시험이 필요하다. 개발 데이터를 한 번만 나눠 연습 시험을 보면 그 한 번의 운에 결과가 좌우된다.

**5-Fold 교차검증**은 개발 데이터를 5조각으로 나눈 뒤,

| 회차 | 학습에 쓰는 조각 | 연습 시험 조각 |
|---|---|---|
| 1 | 2,3,4,5 | 1 |
| 2 | 1,3,4,5 | 2 |
| … | … | … |
| 5 | 1,2,3,4 | 5 |

처럼 5번 돌아가며 시험을 보고 **점수 5개의 평균**을 낸다. 모든 데이터가 한 번씩 시험 문제가 되므로 공정하고 안정적이다.

- `StratifiedKFold`: 각 조각도 유형 비율을 원본과 같게(층화) 나눈다.
- `shuffle=True`: 나누기 전에 순서를 섞는다(파일이 유형별로 정렬돼 있을 수 있으므로).
- `RepeatedStratifiedKFold(n_repeats=3)`: 5-Fold를 **섞는 방식을 바꿔 3번 반복** → 점수 15개. 아주 작은 차이를 비교해야 하는 5단계에서 판단을 더 안정적으로 하기 위해 쓴다.

### 왜 별도 검증셋 대신 교차검증인가?
'학습 60% / 검증 20% / 테스트 20%'로 나누는 방법도 있지만, 데이터가 적을 때는 학습량이 60%로 줄어드는 손해가 크다. 교차검증을 쓰면 개발 데이터 80% 전부를 학습과 검증에 번갈아 활용할 수 있다.

In [ ]:
# ============================================================
# 4. 개발 / 테스트 분할 (80 / 20, 층화)
# ============================================================

# 행 번호(0 ~ 1940)를 나눠서 개발용 번호, 테스트용 번호를 얻는다.
idx_dev, idx_test = train_test_split(
    np.arange(len(df)),
    test_size=0.20,              # 20%를 테스트로
    stratify=y_all,              # 결함 유형 비율 유지
    random_state=RANDOM_STATE,   # 매번 같은 방식으로 나눔 (재현성)
)
y_dev, y_test = y_all[idx_dev], y_all[idx_test]
print(f"\n개발: {len(idx_dev)}개 / 테스트: {len(idx_test)}개")

# 5-Fold 층화 교차검증 (7, 8단계에서 사용)
cv5 = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
# 5-Fold를 3번 반복 = 점수 15개 (5단계에서 사용)
cv_rep = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=RANDOM_STATE)

---
## 5단계. 파생변수를 쓸지 데이터로 결정하기

### 방법
1. **기준 모델(probe)** 하나를 정한다: 기본 설정의 랜덤 포레스트(나무 300그루). 튜닝 전이라 특별히 유리한 설정이 없는 무난한 모델로 비교한다.
2. '원본 변수만' vs '원본 + 파생 3개'로 각각 **반복 교차검증(15회)**을 해서 Macro F1 평균과 표준편차를 구한다.
3. 아래 규칙으로 결정한다.

### 결정 규칙: "평균 향상 > 표준편차의 절반"일 때만 채택
- **표준편차(SD)**: 15번의 점수가 평균에서 얼마나 흩어져 있는지. 즉 '운에 따른 흔들림의 크기'.
- 파생변수를 넣어 점수가 0.002 올랐는데 원래 점수가 회차마다 ±0.02씩 흔들린다면, 그 0.002는 우연일 가능성이 크다.
- 그래서 **흔들림에 비해 의미 있게 좋아질 때만** 채택하고, 애매하면 **더 단순한 쪽(변수가 적은 쪽)**을 고른다. 변수가 적으면 이해·관리가 쉽고 과적합 위험도 줄어든다.
- 기준값 0.5는 엄밀한 통계 검정이 아니라, "작은 차이에 휘둘리지 말자"는 취지의 실용적인 문턱값이다.

### 왜 개발 데이터만 쓰나
이 결정도 '모델을 고르는 행위'이므로 테스트 데이터를 보면 안 된다(`iloc[idx_dev]`).

In [ ]:
# ============================================================
# 5. 파생변수 사용 여부 결정 (개발 데이터 반복 교차검증)
# ============================================================

# 비교용 기준 모델: 기본 설정 랜덤 포레스트, 나무 300그루
#   n_jobs=-1 : 컴퓨터의 CPU 코어를 모두 사용해 빠르게 계산
probe = RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE, n_jobs=-1)
feature_check = {}
for use_extra in [False, True]:
    X_tmp = build_features(df, use_extra).iloc[idx_dev]   # 개발 데이터만 사용
    # 반복 교차검증 15회의 Macro F1 점수 15개
    s = cross_val_score(probe, X_tmp, y_dev, cv=cv_rep, scoring="f1_macro", n_jobs=-1)
    feature_check["원본 + 파생 3개" if use_extra else "원본만"] = (s.mean(), s.std())
    print(f"{'원본 + 파생 3개' if use_extra else '원본만':<12} "
          f"CV Macro F1 = {s.mean():.4f} ± {s.std():.4f}")

# 평균이 표준편차의 절반 이상 좋아질 때만 파생변수 채택 (작은 차이는 단순한 쪽 선택)
gain = feature_check["원본 + 파생 3개"][0] - feature_check["원본만"][0]
USE_EXTRA = gain > 0.5 * feature_check["원본만"][1]
print("→ 파생변수 사용:", USE_EXTRA)

# 결정된 방식으로 최종 입력 표를 만들고 개발/테스트로 나눔
X_all = build_features(df, USE_EXTRA)
feature_cols = X_all.columns.tolist()
X_dev, X_test = X_all.iloc[idx_dev], X_all.iloc[idx_test]

---
## 6단계. 사용할 모델과 설정(하이퍼파라미터) 후보

### 기본 개념: 의사결정나무
'스무고개'와 같다. "면적이 300보다 큰가? → 예 → 밝기 차이가 50보다 큰가? → 아니오 → … → Bumps" 처럼 질문을 이어가며 답을 낸다. 어떤 질문을 어떤 순서로 할지는 데이터에서 자동으로 찾는다.
나무 한 그루는 학습 데이터에 쉽게 과적합되므로, 실제로는 **나무를 수백 그루 모아 함께 판단하게(앙상블)** 한다. 모으는 방식에 따라 두 가족으로 나뉜다.

| 가족 | 방식 | 비유 | 이 코드의 모델 |
|---|---|---|---|
| **배깅(Bagging)** | 각 나무가 데이터·변수의 일부만 무작위로 보고 **독립적으로** 학습 → 투표 | 서로 다른 자료로 공부한 전문가 수백 명의 다수결 | Random Forest, Extra Trees |
| **부스팅(Boosting)** | 나무를 **순서대로** 만들며, 다음 나무가 앞 나무들이 틀린 부분을 집중 보완 | 오답 노트를 계속 고쳐 쓰는 학생 | LightGBM, CatBoost |

### 모델 4종 한 줄 소개
- **Random Forest**: 배깅의 대표. 각 나무가 데이터를 복원추출(같은 행이 여러 번 뽑힐 수 있음)해서 학습하고, 질문할 때마다 변수 일부만 후보로 본다. 튜닝을 대충 해도 성능이 잘 나오는 안정적인 모델.
- **Extra Trees** (Extremely Randomized Trees): Random Forest와 비슷하지만 **질문의 기준값까지 무작위로** 정한다("면적 > 무작위 값"). 나무끼리 더 달라져서 과적합에 강하고 학습이 빠르다.
- **LightGBM**: 마이크로소프트의 부스팅 모델. 가장 오차를 많이 줄이는 가지부터 키우는 방식(leaf-wise)이라 빠르고 정확하지만, 데이터가 적으면 과적합되기 쉬워 복잡도 제한이 중요하다.
- **CatBoost**: 얀덱스의 부스팅 모델. 같은 깊이에서는 모든 가지가 같은 질문을 하는 **대칭 나무**를 쓰고, 과적합을 줄이는 장치가 내장돼 있어 기본 설정만으로도 성능이 좋은 편이다.

### 왜 트리 계열만 쓰나
- 이 데이터처럼 **표 형태의 숫자 데이터**에서 일반적으로 가장 성능이 좋은 계열이다.
- 변수마다 단위가 달라도(픽셀 수 vs 0~1 비율) **크기 맞추기(스케일링)가 필요 없다**.
- "길고 **동시에** 어두우면 긁힘" 같은 **변수 간 조합 관계**를 자동으로 잡는다.
- 실제로 GPT 코드에서 비교했을 때 로지스틱 회귀, SVM, KNN 같은 선형/거리 기반 모델은 트리 계열보다 Macro F1이 0.07~0.17 낮았다.
- 서로 다른 방식의 트리 모델 4개를 고른 이유는 8단계 **앙상블** 때문이다. 방식이 다른 모델은 서로 다른 실수를 하므로 섞으면 실수가 상쇄된다.

---
### 하이퍼파라미터 설명

#### 공통으로 고정한 설정
| 설정 | 의미 | 이렇게 둔 이유 |
|---|---|---|
| `random_state` / `random_seed` = 42 | 무작위 요소의 씨앗 | 결과 재현성 |
| `n_jobs=1` / `thread_count=1` | 모델 하나가 쓰는 CPU 코어 수 | 7단계 탐색 도구가 여러 조합을 **동시에**(n_jobs=-1) 돌리므로, 모델 안에서도 코어를 다 쓰면 서로 자원을 빼앗아 오히려 느려진다(과다 구독). 병렬은 바깥쪽 한 곳에서만. |
| `objective="multiclass"` / `loss_function="MultiClass"` | 정답이 3개 이상인 분류 문제임을 지정 | 결함 유형이 7개 |
| `verbose=-1` / `verbose=0` | 학습 중 로그 출력 끄기 | 화면 정리 |
| `allow_writing_files=False` | CatBoost가 `catboost_info` 폴더를 만들지 않게 | 작업 폴더 정리 |

#### Random Forest / Extra Trees (두 모델 동일한 후보)
| 파라미터 | 의미 | 값을 키우면 | 후보값과 이유 |
|---|---|---|---|
| `n_estimators` | 나무 개수 | 결과가 안정되지만 느려짐. 많다고 과적합되지는 않음 | 300/500/700 — 보통 수백 그루면 성능이 거의 수렴하므로 그 근처만 확인 |
| `max_depth` | 질문을 최대 몇 단계까지 할지 | 세밀해지지만 과적합 위험 | None(제한 없음)/10/15/20/30 — 배깅은 여러 나무 평균으로 과적합이 상쇄돼 제한 없음이 좋을 때가 많지만, 데이터가 작아 제한하는 경우도 비교 |
| `min_samples_split` | 가지를 더 나누려면 최소 몇 개의 샘플이 있어야 하는지 | 덜 쪼개 단순해짐 | 2(기본)/5/9 |
| `min_samples_leaf` | 마지막 잎에 최소 몇 개가 남아야 하는지 | 샘플 1~2개짜리 '외운 잎'을 막음 | 1(기본)/2/4 — 드문 유형(Dirtiness는 개발 데이터에 약 44건)을 지우지 않도록 작은 값 위주 |
| `max_features` | 질문마다 후보로 볼 변수의 비율 | 각 나무는 강해지나 나무끼리 비슷해짐 | "sqrt"(√27≈5개, 기본)/0.3/0.5/0.8 — 다양성과 개별 성능의 균형점을 넓게 탐색 |
| `class_weight` | 유형별로 틀렸을 때의 벌점 가중치 | — | 아래 설명 |

**`class_weight` 자세히** — 클래스 불균형 대응의 핵심이다.
- `None`: 모든 샘플 벌점 동일 → 많은 유형 위주로 학습되기 쉬움.
- `"balanced"`: 가중치 = 전체 개수 ÷ (유형 수 × 그 유형 개수). 개발 데이터 기준으로 대략 Dirtiness ≈ 5.0, Other_Faults ≈ 0.4 → 드문 유형을 틀리면 벌점이 약 12배 크다. 드문 유형의 **재현율**이 올라가는 대신 **정밀도**가 내려갈 수 있다.
- `"balanced_subsample"`: 같은 계산을 각 나무가 뽑은 표본마다 다시 한다(Random Forest 계열 전용).
- 어느 쪽이 Macro F1에 유리한지는 데이터마다 다르므로 **고정하지 않고 튜닝 대상에 포함**했다(GPT 코드에서 가져온 아이디어).

#### LightGBM
| 파라미터 | 의미 | 후보값과 이유 |
|---|---|---|
| `n_estimators` | 부스팅 횟수(나무 개수) | 200~1000 — 아래 학습률과 짝을 이룸 |
| `learning_rate` | 새 나무가 이전 오차를 얼마나 크게 고칠지 | 0.02~0.1 — 작으면 조심스럽게 고치는 대신 나무가 많이 필요. 작은 학습률+많은 나무가 보통 더 정확하지만 느림 |
| `num_leaves` | 나무 한 그루의 최대 잎 개수 (LightGBM 복잡도의 핵심) | 7/15/22/31 — 기본값 31 이하로만. 개발 데이터가 약 1,550건으로 작아 큰 나무는 과적합 위험 |
| `max_depth` | 최대 깊이 (-1 = 제한 없음) | -1/5/7/10 — num_leaves와 함께 복잡도를 이중으로 제한 |
| `min_child_samples` | 잎에 최소 몇 개가 있어야 하는지 | 5/10/20(기본)/33 — 드문 유형 패턴을 잡을 수 있게 기본보다 작은 값도 시험 |
| `subsample` + `subsample_freq` | 각 회차에 데이터 일부만 무작위로 사용 | 0.6/0.8/1.0 — 무작위성으로 과적합 감소. **LightGBM은 `subsample_freq`가 0이면 subsample이 무시**되므로 1로 고정해 실제로 작동하게 함 |
| `colsample_bytree` | 나무마다 변수 일부만 사용 | 0.6/0.8/1.0 — 같은 목적 |
| `reg_lambda` | L2 규제: 잎의 예측값이 극단적으로 커지지 않게 벌점 | 0/0.01/1/5 — 없음부터 강함까지 |
| `class_weight` | 위와 같음 | None/"balanced" |

#### CatBoost
| 파라미터 | 의미 | 후보값과 이유 |
|---|---|---|
| `iterations` | 부스팅 횟수 | 300/500/700 |
| `depth` | 나무 깊이. 대칭 나무라 깊이 d면 잎이 2^d개 | 4~7 (잎 16~128개) — 기본값 6 주변 |
| `learning_rate` | 학습률 | 0.03/0.05/0.1 |
| `l2_leaf_reg` | L2 규제 강도 | 1/3(기본)/5/7 |
| `auto_class_weights` | 클래스 가중치 자동 계산 | None/"Balanced"/"SqrtBalanced" — SqrtBalanced는 Balanced 가중치에 제곱근을 씌운 **중간 강도**(Dirtiness ≈ 2.2배) |

> 후보값을 고른 공통 원칙: 각 라이브러리의 **기본값을 중심으로**, 과적합을 막는 쪽(더 단순)과 표현력을 높이는 쪽(더 복잡)을 모두 포함하도록 넓게 잡았다. 데이터가 작으므로 '복잡한 쪽'은 기본값을 크게 넘지 않게 했다.

In [ ]:
# ============================================================
# 6. 모델 및 탐색 공간
# ============================================================
# GPT 코드의 기본 비교에서 선형/거리 기반 모델(로지스틱, SVM, KNN)은
# 트리 계열보다 Macro F1이 0.07~0.17 낮았으므로 트리 계열만 사용한다.
# class_weight는 모든 모델에서 튜닝 대상이다.
# 모델 내부 병렬(n_jobs/thread_count)은 1로 두고, 병렬 처리는 7단계 탐색 도구에 맡긴다.
# ============================================================

models = {
    # 배깅 계열
    "Random Forest": RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=1),
    "Extra Trees": ExtraTreesClassifier(random_state=RANDOM_STATE, n_jobs=1),
    # 부스팅 계열
    "LightGBM": LGBMClassifier(objective="multiclass",    # 다중 분류
                               random_state=RANDOM_STATE,
                               n_jobs=1, verbose=-1),     # 로그 끄기
    "CatBoost": CatBoostClassifier(loss_function="MultiClass",   # 다중 분류
                                   random_seed=RANDOM_STATE,
                                   verbose=0,                   # 로그 끄기
                                   allow_writing_files=False,   # 기록 파일 생성 안 함
                                   thread_count=1),
}

# 모델별로 시험해 볼 설정 후보 (의미와 이유는 위 표 참고)
param_spaces = {
    "Random Forest": {
        "n_estimators": [300, 500, 700],                 # 나무 개수
        "max_depth": [None, 10, 15, 20, 30],             # 최대 깊이 (None=제한 없음)
        "min_samples_split": [2, 5, 9],                  # 분할에 필요한 최소 샘플 수
        "min_samples_leaf": [1, 2, 4],                   # 잎의 최소 샘플 수
        "max_features": ["sqrt", 0.3, 0.5, 0.8],         # 분할마다 볼 변수 비율
        "class_weight": [None, "balanced", "balanced_subsample"],  # 드문 유형 가중치
    },
    "Extra Trees": {
        "n_estimators": [300, 500, 700],
        "max_depth": [None, 10, 15, 20, 30],
        "min_samples_split": [2, 5, 9],
        "min_samples_leaf": [1, 2, 4],
        "max_features": ["sqrt", 0.3, 0.5, 0.8],
        "class_weight": [None, "balanced", "balanced_subsample"],
    },
    "LightGBM": {
        "n_estimators": [200, 400, 700, 1000],           # 부스팅 횟수
        "learning_rate": [0.02, 0.03, 0.05, 0.1],        # 한 번에 고치는 정도
        "num_leaves": [7, 15, 22, 31],                   # 나무당 최대 잎 수 (복잡도)
        "max_depth": [-1, 5, 7, 10],                     # 최대 깊이 (-1=제한 없음)
        "min_child_samples": [5, 10, 20, 33],            # 잎의 최소 샘플 수
        "subsample": [0.6, 0.8, 1.0],                    # 회차마다 쓸 데이터 비율
        "subsample_freq": [1],                           # subsample이 실제로 작동하게 함
        "colsample_bytree": [0.6, 0.8, 1.0],             # 나무마다 쓸 변수 비율
        "reg_lambda": [0, 0.01, 1, 5],                   # L2 규제 강도
        "class_weight": [None, "balanced"],
    },
    "CatBoost": {
        "iterations": [300, 500, 700],                   # 부스팅 횟수
        "depth": [4, 5, 6, 7],                           # 대칭 나무 깊이 (잎 2^depth개)
        "learning_rate": [0.03, 0.05, 0.1],
        "l2_leaf_reg": [1, 3, 5, 7],                     # L2 규제 강도
        "auto_class_weights": [None, "Balanced", "SqrtBalanced"],  # 클래스 가중치
    },
}

---
## 7단계. 하이퍼파라미터 튜닝 — 무작위 탐색 + 5-Fold 교차검증

### 왜 '무작위' 탐색인가?
Random Forest 후보만 해도 3×5×3×3×4×3 = **1,620가지** 조합이고, LightGBM은 수만 가지다. 모든 조합을 다 해 보는 **격자 탐색(Grid Search)**은 시간이 너무 오래 걸린다.
**무작위 탐색(Randomized Search)**은 그중 `N_ITER`(=20)개를 무작위로 골라 시험한다. 실제로 성능에 큰 영향을 주는 파라미터는 몇 개뿐인 경우가 많아서, 무작위로 뽑아도 좋은 조합을 꽤 효율적으로 찾는다는 것이 알려져 있다.

### 각 조합은 어떻게 평가하나?
조합 하나마다 **5-Fold 교차검증**으로 Macro F1 평균을 구한다. 모델당 20조합 × 5회 = **100번 학습**.
단일 검증셋 하나로 평가하면 그 검증셋에 우연히 잘 맞는 조합이 뽑힐 수 있는데, 5번 평균을 내면 그런 우연이 줄어든다(GPT 코드에서 가져온 방식).

### 코드 속 설정
- `scoring="f1_macro"`: 주 지표인 Macro F1로 비교.
- `n_jobs=-1`: 여러 조합·회차를 CPU 코어 전부로 동시에 계산.
- `refit=False`: 탐색이 끝난 뒤 최적 조합으로 모델을 자동으로 다시 학습하지 않음. 여기서는 **최적 설정값만** 필요하고, 실제 학습은 8·9단계에서 직접 하므로 시간을 아낀다.
- `clone(model).set_params(**best_params_)`: 학습되지 않은 빈 모델을 복사해 최적 설정만 입혀 둔다.
- 결과표의 `CV SD`는 5회 점수의 표준편차. 모델 간 점수 차이가 이 값보다 작으면 사실상 비슷한 성능이라고 봐야 한다.

In [ ]:
# ============================================================
# 7. 하이퍼파라미터 튜닝 (개발 데이터 5-Fold, Macro F1)
# ============================================================

tuned = {}        # 모델 이름 → 최적 설정을 입힌 (아직 학습 안 된) 모델
tuning_rows = []  # 결과표용

for name, model in models.items():
    print(f"\n{name} 튜닝 중... ({N_ITER}개 조합 × 5-Fold)")
    start = time.time()
    search = RandomizedSearchCV(
        model,                      # 튜닝할 모델
        param_spaces[name],         # 설정 후보
        n_iter=N_ITER,              # 무작위로 시험할 조합 수
        scoring="f1_macro",         # 평가 기준
        cv=cv5,                     # 5-Fold 층화 교차검증
        random_state=RANDOM_STATE,  # 어떤 조합을 뽑을지 고정
        n_jobs=-1,                  # CPU 코어 전부 사용
        refit=False,                # 최적값만 얻고 재학습은 생략
    )
    search.fit(X_dev, y_dev)        # 개발 데이터만 사용
    tuned[name] = clone(model).set_params(**search.best_params_)

    tuning_rows.append({
        "Model": name,
        "CV Macro F1": search.best_score_,   # 최적 조합의 5회 평균 점수
        "CV SD": search.cv_results_["std_test_score"][search.best_index_],  # 5회 점수의 흔들림
        "Best Params": search.best_params_,
        "Time(s)": round(time.time() - start, 1),
    })
    print(f"  CV Macro F1 = {search.best_score_:.4f} | {search.best_params_}")

# 점수 높은 순으로 정렬 → 이 순위가 8단계 앙상블 후보 구성에 쓰인다
tuning_df = (pd.DataFrame(tuning_rows)
             .sort_values("CV Macro F1", ascending=False).reset_index(drop=True))
print("\n[튜닝 결과]")
display(tuning_df)

---
## 8단계. Out-of-fold 예측, 앙상블, 클래스별 확률 보정

이 단계에서는 "모델 하나를 쓸까, 여러 개를 섞을까?", "드문 유형을 얼마나 적극적으로 예측할까?"를 결정한다. 이 결정에도 테스트 데이터는 쓰지 않는다.

### (1) 예측 확률(predict_proba)
모델은 답 하나만 내는 게 아니라 **유형별 확률**을 낸다. 예: `Bumps 0.45, Other_Faults 0.30, Dirtiness 0.15, …` (합계 1). 보통은 가장 높은 것(argmax)을 답으로 한다.

### (2) Out-of-fold(OOF) 예측이란?
개발 데이터 각 행에 대해 **"그 행을 학습에 쓰지 않은 모델"이 낸 예측 확률**이다. 5-Fold로 나눠, 각 조각은 나머지 4조각으로 학습한 모델이 예측한다. 그러면 개발 데이터 약 1,550건 전부에 대해 '처음 보는 문제를 풀었을 때의 답안지'가 생긴다.
→ 이 답안지로 앙상블 조합과 확률 보정 값을 정하면, 테스트를 건드리지 않고도 공정하게 비교할 수 있다.

> 단, 7단계 튜닝에 썼던 같은 조각 구성으로 만든 답안지라 점수가 **약간 낙관적**일 수 있다. 그래서 최종 판단은 10단계 테스트로 한다.

### (3) 소프트 보팅 앙상블
여러 모델의 **확률을 평균**내서 답을 정한다(soft voting). 확률을 쓰기 때문에 "A 모델은 아주 확신, B 모델은 애매"한 정도까지 반영된다.
서로 방식이 다른 모델은 다른 곳에서 실수하므로, 평균을 내면 실수가 상쇄돼 보통 단일 모델보다 안정적이다.
비교 후보: 단일 모델 4개 + 7단계 순위 상위 2개 / 상위 3개 / 전체 4개 평균 = **총 7개 후보**.

### (4) 클래스별 확률 보정
`class_weight`로 학습 단계에서 이미 불균형을 다뤘지만, **예측 단계에서 한 번 더 미세 조정**할 수 있다. 각 유형의 확률에 **배율**을 곱한 뒤 가장 큰 것을 답으로 한다.

예: 확률이 `Bumps 0.40, Dirtiness 0.35`일 때 Dirtiness 배율이 1.3이면 0.35×1.3 = 0.455 → 답이 Dirtiness로 바뀐다.
- 배율 > 1: 그 유형을 더 자주 예측 → 재현율↑, 정밀도↓
- 배율 < 1: 덜 자주 예측 → 재현율↓, 정밀도↑

**배율을 찾는 방법(좌표 탐색)**: 모든 배율을 1로 시작 → 첫 번째 유형의 배율만 0.5, 0.6, …, 3.0으로 바꿔 보며 OOF Macro F1이 가장 높은 값으로 바꿈 → 두 번째 유형 … → 7번째까지 한 바퀴. 이걸 최대 3바퀴(더 좋아지지 않으면 중단) 반복한다.
- 범위 0.5~3.0: 어떤 유형을 절반으로 낮추는 것부터 3배로 높이는 것까지면 충분하고, 그 이상은 과하게 맞추는 위험이 크다.

**왜 이 방법인가?** GPT 코드는 드문 유형을 보수적으로(정밀도 위주), Claude 코드는 적극적으로(재현율 위주) 예측하는 경향이 있었다. 둘 중 어느 쪽이 맞는지 **사람이 정하지 않고, Macro F1이 가장 높아지는 지점을 데이터가 찾게** 한 것이다.

In [ ]:
# ============================================================
# 8. Out-of-fold 확률 예측
# ============================================================
# 개발 데이터의 모든 행에 대해 "그 행을 학습에 쓰지 않은 모델"의 예측 확률을 얻는다.
# 앙상블 조합과 클래스별 확률 보정은 이 확률로만 결정하고 테스트는 건드리지 않는다.
# (튜닝에 쓴 fold와 같으므로 약간 낙관적일 수 있음 → 최종 판단은 테스트에서)
# ============================================================

oof = {}   # 모델 이름 → (개발 데이터 행 수 × 7) 확률 표
for name, est in tuned.items():
    oof[name] = cross_val_predict(
        clone(est), X_dev, y_dev, cv=cv5,
        method="predict_proba",   # 답 대신 유형별 확률을 받음
        n_jobs=-1,
    )


def tune_class_scale(P, y, grid=np.round(np.arange(0.5, 3.01, 0.1), 2), n_rounds=3):
    """
    예측 확률에 클래스별 배율을 곱해 argmax 했을 때 Macro F1이 최대가 되는 배율 탐색.
    배율 > 1 이면 그 클래스를 더 자주 예측(재현율↑ 정밀도↓), < 1 이면 반대.
    → GPT 코드의 '보수적 예측'과 Claude 코드의 '소수 클래스 적극 예측' 사이 균형점을
      사람이 정하지 않고 데이터로 찾는다.

    P: OOF 확률 표, y: 실제 정답
    grid: 시험할 배율 후보 0.5, 0.6, ..., 3.0
    n_rounds: 7개 유형을 한 바퀴씩 도는 횟수의 최대값
    """
    w = np.ones(P.shape[1])                       # 시작: 모든 배율 1 (보정 없음)
    best = macro_f1(y, P.argmax(axis=1))          # 보정 전 점수
    for _ in range(n_rounds):
        improved = False
        for k in range(P.shape[1]):               # 유형 하나씩
            for g in grid:                        # 배율 후보 하나씩
                trial = w.copy()
                trial[k] = g
                s = macro_f1(y, (P * trial).argmax(axis=1))
                if s > best + 1e-9:               # 실제로 좋아졌을 때만 채택
                    best, w, improved = s, trial, True
        if not improved:                          # 한 바퀴 동안 개선 없으면 종료
            break
    return w, best


# 앙상블 후보 구성 (7단계 튜닝 순위 기준)
ranked = tuning_df["Model"].tolist()
candidates = {name: [name] for name in ranked}   # 단일 모델 4개
candidates["앙상블(상위 2)"] = ranked[:2]
candidates["앙상블(상위 3)"] = ranked[:3]
candidates["앙상블(전체 4)"] = ranked

cand_rows = []
cand_scale = {}
for cname, members in candidates.items():
    P = np.mean([oof[m] for m in members], axis=0)   # 구성 모델들의 확률 평균 (소프트 보팅)
    raw = macro_f1(y_dev, P.argmax(axis=1))           # 보정 전 점수
    w, scaled = tune_class_scale(P, y_dev)            # 보정 배율과 보정 후 점수
    cand_scale[cname] = w
    cand_rows.append({"Candidate": cname, "OOF Macro F1": raw,
                      "OOF Macro F1 (확률 보정)": scaled})

cand_df = (pd.DataFrame(cand_rows)
           .sort_values("OOF Macro F1 (확률 보정)", ascending=False).reset_index(drop=True))
print("\n[후보별 Out-of-fold 성능]")
display(cand_df.round(4))

---
## 9단계. 최종 모델 결정

### 선택 규칙
1. 8단계 표에서 **보정 후 OOF 점수가 가장 높은 후보**를 고른다.
2. 확률 보정은 OOF 답안지에 **딱 맞춰서 찾은 값**이라, 그 답안지에서의 점수 향상은 실제보다 부풀려져 있다(약한 과적합). 그래서 **향상폭이 0.005 미만이면 보정을 쓰지 않는다**. 작은 이득을 위해 복잡함과 과적합 위험을 감수하지 않겠다는 실용적인 문턱이다.

### `ScaledSoftVoting` — 직접 만든 모델 클래스
"여러 모델의 확률 평균 + 클래스별 배율"을 하나의 모델처럼 쓰기 위해 만들었다. scikit-learn의 규칙(`fit`, `predict`, `predict_proba`)을 따르기 때문에 12단계 변수 중요도 계산 같은 도구에 그대로 넣을 수 있다.
- `fit`: 구성 모델들을 각각 복제해서 학습.
- `predict_proba`: 각 모델의 확률을 평균 → 배율을 곱함 → **다시 합이 1이 되도록 나눔**(그래야 '확신도'로 해석 가능).
- `predict`: 확률이 가장 큰 유형을 답으로.

마지막으로 고른 구성으로 **개발 데이터 전체(80%)**를 학습시킨다. 교차검증 때는 개발 데이터의 4/5만 썼지만, 이제 결정이 끝났으므로 개발 데이터를 다 쓴다.

In [ ]:
# ============================================================
# 9. 최종 후보 선택
# ============================================================
# 확률 보정은 OOF에 맞춰 찾은 값이라 그 자체로 약간 낙관적이다.
# 보정 이득이 0.005 미만이면 보정 없이 사용한다.
# ============================================================

class ScaledSoftVoting(BaseEstimator, ClassifierMixin):
    """여러 모델의 확률 평균 + 클래스별 배율 보정
    BaseEstimator, ClassifierMixin을 상속하면 scikit-learn 도구들과 호환된다."""

    def __init__(self, estimators, class_scale=None):
        self.estimators = estimators     # 구성 모델 목록 (학습 전)
        self.class_scale = class_scale   # 유형별 배율 (None이면 보정 없음)

    def fit(self, X, y):
        # 원본을 건드리지 않도록 복제해서 각각 학습
        self.estimators_ = [clone(e).fit(X, y) for e in self.estimators]
        self.classes_ = np.unique(y)     # 유형 번호 목록 (0~6)
        return self

    def predict_proba(self, X):
        P = np.mean([e.predict_proba(X) for e in self.estimators_], axis=0)  # 확률 평균
        if self.class_scale is not None:
            P = P * np.asarray(self.class_scale)   # 유형별 배율 적용
            P = P / P.sum(axis=1, keepdims=True)   # 합이 1이 되게 다시 맞춤
        return P

    def predict(self, X):
        return self.classes_[self.predict_proba(X).argmax(axis=1)]   # 확률 최대 유형


best_row = cand_df.iloc[0]               # 보정 후 OOF 점수 1위 후보
best_cand = best_row["Candidate"]
# 보정으로 0.005 이상 좋아질 때만 보정 사용
use_scale = (best_row["OOF Macro F1 (확률 보정)"] - best_row["OOF Macro F1"]) >= 0.005
final_scale = cand_scale[best_cand] if use_scale else None

print("\n최종 후보:", best_cand, "| 구성:", candidates[best_cand])
print("확률 보정 사용:", use_scale)
if use_scale:
    display(pd.Series(final_scale, index=FAULT_COLS, name="클래스 배율").round(2))

# 개발 데이터 전체로 최종 모델 학습
final_model = ScaledSoftVoting(
    [tuned[m] for m in candidates[best_cand]], class_scale=final_scale
).fit(X_dev, y_dev)

---
## 10단계. 테스트 평가 (단 한 번)

드디어 금고에서 테스트 데이터를 꺼낸다. 여기서 나온 점수가 **"처음 보는 강판에 대한 예상 성능"**이다.

### 부트스트랩 95% 신뢰구간
테스트가 389건뿐이라, 어떤 결함들이 테스트에 들어갔는지(운)에 따라 점수가 꽤 달라질 수 있다. 이 흔들림의 크기를 추정하는 방법이 **부트스트랩**이다.
1. 테스트 389건에서 **복원추출**(같은 건이 여러 번 뽑힐 수 있게)로 389건을 다시 뽑아 '가상의 테스트셋'을 만든다.
2. 그 가상 테스트셋의 Macro F1을 계산한다.
3. 1~2를 1,000번 반복 → 점수 1,000개.
4. 하위 2.5%~상위 2.5%를 잘라낸 구간 = **95% 신뢰구간**.

해석: 예를 들어 구간이 [0.78, 0.86]이면 "테스트셋 구성의 운에 따라 점수가 이 정도 범위에서 움직일 수 있다"는 뜻이다. 다른 코드와 점수 차이가 0.01이라면, 이 구간 폭(0.08)에 비해 너무 작아서 **어느 쪽이 낫다고 단정할 수 없다**.

### Classification Report 읽는 법
유형별로 `precision`(정밀도), `recall`(재현율), `f1-score`, `support`(테스트에 있는 실제 개수)가 나온다. `support`가 작은 유형(Dirtiness 11건 등)의 점수는 1~2건 차이로도 크게 변하므로 너무 확대 해석하지 않는다.

### 참고용 후보별 테스트 점수
다른 후보들이 테스트에서 몇 점이었는지 **구경만** 한다. 이걸 보고 최종 선택을 바꾸면, 테스트를 '모델 고르기'에 쓴 것이 되어 테스트 점수가 더 이상 공정한 성능 추정이 아니게 된다.

In [ ]:
# ============================================================
# 10. 테스트 평가 (마지막 1회)
# ============================================================

y_pred = final_model.predict(X_test)
test_scores = score_row(y_test, y_pred)

print("\n" + "=" * 60)
print(f"최종 테스트 결과 - {best_cand}{' + 확률 보정' if use_scale else ''}")
print("=" * 60)
display(pd.DataFrame([test_scores]).round(4))

# 부트스트랩 95% 신뢰구간: 테스트셋 구성에 따라 점수가 얼마나 흔들리는지 확인
rng = np.random.default_rng(RANDOM_STATE)
boot = []
for _ in range(1000):                                  # 1,000번 반복
    b = rng.integers(0, len(y_test), len(y_test))      # 복원추출로 뽑은 행 번호
    boot.append(macro_f1(y_test[b], y_pred[b]))        # 그 가상 테스트셋의 점수
lo, hi = np.percentile(boot, [2.5, 97.5])              # 양 끝 2.5%씩 제외
print(f"테스트 Macro F1 95% 신뢰구간: [{lo:.3f}, {hi:.3f}]")
print("  → 다른 코드와의 점수 차이가 이 구간 폭보다 작으면 우열을 단정하기 어렵다.")

print("\n[Classification Report - Test]")
print(classification_report(y_test, y_pred, labels=LABELS,
                            target_names=FAULT_COLS, digits=4, zero_division=0))

# 참고용: 다른 후보들의 테스트 성능 (이것을 보고 선택을 바꾸지 않는다)
ref_rows = []
for cname, members in candidates.items():
    m = ScaledSoftVoting([tuned[x] for x in members]).fit(X_dev, y_dev)
    ref_rows.append({"Candidate": cname, "Test Macro F1 (보정 없음)":
                     macro_f1(y_test, m.predict(X_test))})
print("[참고] 후보별 테스트 Macro F1")
display(pd.DataFrame(ref_rows).round(4))

---
## 11단계. 혼동행렬과 오분류 분석

### 혼동행렬(Confusion Matrix)이란?
행 = 실제 유형, 열 = 예측 유형인 표. **대각선**에 있는 칸은 맞힌 개수, 나머지는 틀린 개수다. 예를 들어 '실제 Pastry 행, 예측 Other_Faults 열'의 숫자가 크면 "Pastry를 Other_Faults로 자주 착각한다"는 뜻이다.

- 왼쪽 그림: 개수 그대로.
- 오른쪽 그림(`normalize="true"`): 각 행을 그 유형의 실제 개수로 나눈 비율. 대각선 값이 곧 그 유형의 **재현율**이다. 개수가 적은 유형도 같은 기준으로 비교할 수 있다.

### 왜 이 분석을 하나?
점수 하나만으로는 **어디서** 틀리는지 모른다. 자주 헷갈리는 쌍을 알면
- 그 두 유형을 구분하는 새 변수를 고민하거나,
- 현장에서 "이 두 유형으로 예측되면 사람이 한 번 더 확인" 같은 운영 규칙을 만들 수 있다.

`Other_Faults`는 '기타'라는 이름처럼 여러 모양이 섞여 있어서 다른 유형과 헷갈리기 쉬운 경우가 많다.
`Confidence`(가장 높은 확률값)를 같이 보면, 틀린 예측이 '애매하게 틀린 것'인지 '확신하며 틀린 것'인지도 알 수 있다.

In [ ]:
# ============================================================
# 11. 혼동행렬 및 오분류 분석
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(18, 7))
# 왼쪽: 개수
ConfusionMatrixDisplay(confusion_matrix(y_test, y_pred, labels=LABELS),
                       display_labels=FAULT_COLS).plot(
    ax=axes[0], xticks_rotation=45, colorbar=False)
axes[0].set_title("혼동행렬 (개수)")
# 오른쪽: 실제 유형별 비율 (대각선 = 재현율)
ConfusionMatrixDisplay(confusion_matrix(y_test, y_pred, labels=LABELS, normalize="true"),
                       display_labels=FAULT_COLS).plot(
    ax=axes[1], xticks_rotation=45, colorbar=False, values_format=".2f")
axes[1].set_title("혼동행렬 (실제 유형 기준 비율)")
for ax in axes:
    ax.set_xlabel("예측 결함 유형")
    ax.set_ylabel("실제 결함 유형")
plt.tight_layout()
plt.show()

# 테스트 각 행의 실제값, 예측값, 확신도(가장 높은 확률)
test_result = pd.DataFrame({
    "Actual": [int_to_class[i] for i in y_test],
    "Predicted": [int_to_class[i] for i in y_pred],
    "Confidence": final_model.predict_proba(X_test).max(axis=1),
}, index=X_test.index)
misclassified = test_result[test_result["Actual"] != test_result["Predicted"]]

print(f"\n오분류: {len(misclassified)} / {len(y_test)} ({len(misclassified) / len(y_test):.2%})")
# 가장 자주 나온 '실제 → 예측' 착각 조합 상위 10개
display(misclassified.groupby(["Actual", "Predicted"]).size()
        .sort_values(ascending=False).head(10).to_frame("Count"))

---
## 12단계. 어떤 변수가 중요했나 — Permutation Importance

### 아이디어
"이 변수가 없으면 모델이 얼마나 못하게 되나?"를 잰다.
1. 테스트 데이터에서 변수 하나(예: 면적)의 값만 **행끼리 무작위로 섞는다**. 그러면 그 변수는 정답과의 관계가 끊겨 쓸모없는 정보가 된다(다른 변수는 그대로).
2. 섞은 데이터로 Macro F1을 다시 잰다.
3. **원래 점수 − 섞은 후 점수 = 중요도**. 많이 떨어질수록 모델이 그 변수에 많이 의존한다는 뜻.
4. 섞는 방식에 따라 결과가 흔들리므로 `n_repeats=20`번 반복해 평균과 표준편차를 낸다(그래프의 오차 막대).

### 왜 이 방법인가?
- 최종 모델은 여러 모델의 앙상블이라 모델 내장 중요도(`feature_importances_`)가 하나로 정해지지 않는다. Permutation Importance는 **모델 종류와 상관없이** 쓸 수 있다.
- 트리 내장 중요도는 값 종류가 많은 변수를 과대평가하는 경향이 있는데, 이 방법은 실제 **성능 하락**을 직접 재므로 그런 편향이 적다.
- **테스트 데이터**로 재기 때문에 "처음 보는 데이터를 맞힐 때 실제로 쓰이는 변수"를 보여 준다.

### 해석할 때 주의
서로 비슷한 정보를 가진 변수가 여러 개면(예: 면적과 둘레), 하나를 섞어도 다른 것이 대신해 주므로 **둘 다 중요도가 낮게** 나올 수 있다. "중요도가 낮다 = 쓸모없다"로 단정하면 안 된다. 중요도가 0 근처나 음수인 변수는 이 모델에서 거의 기여하지 않는다는 뜻이다.

In [ ]:
# ============================================================
# 12. 변수 중요도 (Permutation Importance, 테스트 데이터)
# ============================================================

perm = permutation_importance(
    final_model, X_test, y_test,
    scoring="f1_macro",          # 섞기 전후 Macro F1 차이로 중요도 측정
    n_repeats=20,                # 변수마다 20번 섞어 평균
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
importance_df = (pd.DataFrame({
    "Feature": feature_cols,
    "Importance Mean": perm.importances_mean,   # 평균 점수 하락량
    "Importance SD": perm.importances_std,      # 20번 사이의 흔들림
}).sort_values("Importance Mean", ascending=False).reset_index(drop=True))
display(importance_df.head(15).round(4))

# 상위 15개 막대그래프 (위에서부터 중요한 순)
top = importance_df.head(15).iloc[::-1]
plt.figure(figsize=(9, 7))
plt.barh(top["Feature"], top["Importance Mean"], xerr=top["Importance SD"])
plt.xlabel("섞었을 때 Macro F1 감소량")
plt.title("Permutation Importance 상위 15개")
plt.tight_layout()
plt.show()

---
## 13단계. 전체 데이터로 다시 학습하고 저장

### 왜 테스트 데이터까지 넣어서 다시 학습하나?
평가는 10단계에서 이미 끝났다. 실제로 공장에 쓸 모델은 **데이터를 하나라도 더 본 쪽**이 보통 더 좋다. 그래서 같은 설정(구성 모델, 하이퍼파라미터, 배율)을 그대로 두고 1,941건 전부로 다시 학습한다.
- 이 모델의 성능은 따로 잴 데이터가 없으므로, **10단계 테스트 점수를 성능의 대표값**으로 보고한다(데이터가 조금 더 많으니 실제로는 그와 비슷하거나 약간 나을 것으로 기대).

### 무엇을 저장하나?
모델만 저장하면 나중에 "파생변수를 썼나? 변수 순서는?"을 알 수 없어서 예측이 틀어진다. 그래서 **모델 + 사용한 설정 + 변수 목록 + 유형 이름 + 테스트 점수**를 한 묶음(dict)으로 저장한다.

In [ ]:
# ============================================================
# 13. 전체 데이터로 재학습 및 저장
# ============================================================

deploy_model = ScaledSoftVoting(
    [tuned[m] for m in candidates[best_cand]], class_scale=final_scale
).fit(X_all, y_all)    # 개발 + 테스트 전체

joblib.dump({
    "model": deploy_model,                                    # 학습된 모델
    "candidate": best_cand,                                   # 선택된 후보 이름
    "members": candidates[best_cand],                         # 구성 모델
    "params": {m: tuned[m].get_params() for m in candidates[best_cand]},  # 하이퍼파라미터
    "class_scale": None if final_scale is None else final_scale.tolist(), # 유형별 배율
    "use_extra_features": USE_EXTRA,                          # 파생변수 사용 여부
    "feature_cols": feature_cols,                             # 입력 변수 순서
    "class_names": FAULT_COLS,                                # 유형 이름 (번호 순)
    "test_scores": test_scores,                               # 성능 대표값
}, "steel_fault_model_combined.joblib")
print("\n저장 완료: steel_fault_model_combined.joblib")

---
## 14단계. 새 데이터 예측 함수

저장한 모델 파일을 불러와 새 강판 데이터의 결함 유형을 예측한다.

### 처리 순서
1. 모델 묶음 불러오기
2. 입력에 결함 정답 열이 있으면 제거(새 데이터에는 보통 없지만 실수 방지)
3. 1단계와 같은 방식으로 강종 열 병합 → 원본 형식(`TypeOfSteel_A300/A400`)이든 병합 형식(`Steel_Type`)이든 받을 수 있음
4. 3단계와 같은 방식으로 입력 변수 생성 → **학습 때와 같은 열 순서**로 정렬
5. 유형별 확률, 최종 예측 유형, 확신도를 표로 반환

### 주의
- 저장 파일 안에 `ScaledSoftVoting` 모델이 들어 있으므로, 불러오는 쪽 코드에도 **이 클래스 정의가 있어야** 한다(파이썬 저장 방식의 특성).
- 아래 데모는 테스트 데이터 5건을 넣어 보지만, 13단계 모델은 이 데이터까지 학습했으므로 **잘 맞아도 성능 증거가 아니다**. 함수가 제대로 돌아가는지 확인하는 용도다.
- `confidence`가 낮은 예측(예: 0.5 미만)은 현장에서 사람이 한 번 더 확인하는 식으로 활용할 수 있다.

In [ ]:
# ============================================================
# 14. 새 데이터 예측 함수
# ============================================================
# 원본 형식(TypeOfSteel_A300/A400) 또는 병합 형식(Steel_Type) 모두 입력 가능.
# ScaledSoftVoting 클래스가 정의된 환경에서 불러와야 한다.
# ============================================================

def predict_faults(new_data: pd.DataFrame,
                   model_path: str = "steel_fault_model_combined.joblib") -> pd.DataFrame:
    bundle = joblib.load(model_path)                       # ① 모델 묶음 불러오기
    names = bundle["class_names"]
    # ② 정답 열이 섞여 있으면 제거  ③ 강종 열 병합
    merged = merge_columns(new_data.drop(columns=names, errors="ignore"))
    # ④ 학습 때와 같은 방식·같은 열 순서로 입력 생성
    X_new = build_features(merged, bundle["use_extra_features"])[bundle["feature_cols"]]

    # ⑤ 유형별 확률 + 예측 유형 + 확신도
    proba = bundle["model"].predict_proba(X_new)
    out = pd.DataFrame(proba, columns=[f"prob_{n}" for n in names], index=new_data.index)
    out.insert(0, "predicted_fault", [names[i] for i in proba.argmax(axis=1)])
    out.insert(1, "confidence", proba.max(axis=1))
    return out


demo = predict_faults(raw_df.iloc[idx_test[:5]])
demo.insert(0, "actual_fault", df.iloc[idx_test[:5]][TARGET].values)
print("\n[예측 함수 동작 확인 - 전체 데이터로 재학습한 모델이므로 성능 지표 아님]")
display(demo.round(3))

---
## 정리: 사용한 기법과 선택 근거 한눈에 보기

| 단계 | 기법 | 한 줄 설명 | 선택 근거 |
|---|---|---|---|
| 0 | Macro F1 | 유형별 F1의 단순 평균 | 클래스 불균형에서 드문 유형도 똑같이 평가 |
| 1 | 원-핫 구조 검증 | 각 행에 1이 정확히 하나인지 확인 | 병합 오류를 조기에 차단 |
| 2 | 카이제곱 검정, Cramér's V | 두 범주형 변수의 관련 여부와 강도 | 강종이 결함 예측 단서인지 확인 |
| 3 | 파생변수 | 길쭉함, 밝기 대비, 둘레/면적 | 트리가 스스로 만들기 어려운 '두 변수 조합' 정보 |
| 4 | 층화 분할, 5-Fold 교차검증 | 비율 유지하며 나누기, 돌아가며 연습 시험 | 작은 데이터에서 공정·안정적인 평가 |
| 5 | 반복 교차검증 + 0.5 SD 규칙 | 파생변수 채택 여부 결정 | 우연한 작은 차이에 휘둘리지 않고 단순한 쪽 선호 |
| 6 | RF, ET, LightGBM, CatBoost | 배깅 2종 + 부스팅 2종 | 표 데이터에 강하고, 서로 달라 앙상블 효과 |
| 6 | class_weight 튜닝 | 드문 유형 벌점 가중 | 불균형 대응 강도를 데이터가 결정 |
| 7 | 무작위 탐색 | 조합 일부만 무작위로 시험 | 격자 탐색 대비 시간 효율 |
| 8 | Out-of-fold 예측 | 각 행을 안 본 모델의 예측 | 테스트 없이 공정하게 앙상블·보정 결정 |
| 8 | 소프트 보팅 | 확률 평균 | 모델 간 실수 상쇄 |
| 8 | 클래스별 확률 보정 | 유형별 확률 배율 | 정밀도/재현율 균형을 데이터가 결정 |
| 10 | 부트스트랩 신뢰구간 | 테스트 재추출 1,000번 | 점수가 운에 따라 얼마나 흔들리는지 확인 |
| 11 | 혼동행렬 | 실제 × 예측 표 | 어떤 유형끼리 헷갈리는지 파악 |
| 12 | Permutation Importance | 변수를 섞어 성능 하락 측정 | 앙상블에도 적용 가능, 편향 적음 |
| 13 | 전체 데이터 재학습 | 평가 후 모든 데이터로 학습 | 배포 모델의 학습량 극대화 |

### 이 코드의 한계 (알고 쓰기)
- 8단계 OOF 답안지가 7단계 튜닝과 같은 조각 구성이라 OOF 점수는 약간 낙관적이다(최종 판단은 테스트로 보완).
- 5단계의 0.5 SD, 9단계의 0.005 같은 문턱값은 엄밀한 통계 기준이 아니라 실용적인 경험 규칙이다.
- 테스트가 389건이라 드문 유형(Dirtiness 약 11건, Stains 약 14건)의 유형별 점수는 몇 건 차이로도 크게 변한다. 신뢰구간과 함께 해석해야 한다.